<a href="https://colab.research.google.com/github/geeta781/student-mental-health-app/blob/main/step2_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STEP 2 — EDA (Easy Data Augmentation)
**Input:** step0_extended.csv from Google Drive
**Output:** step2_eda.csv to Google Drive
**Time:** ~20 min | **GPU:** Not required

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/emotion_augmentation/'
os.makedirs(DRIVE, exist_ok=True)
print('Drive mounted:', DRIVE)


Mounted at /content/drive
Drive mounted: /content/drive/MyDrive/emotion_augmentation/


In [ ]:
import subprocess, sys
for p in ['nltk','sentence-transformers','scikit-learn','tqdm']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
print('Packages ready')


Packages ready


In [ ]:
import pandas as pd, numpy as np, random, warnings, string
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import wordnet, stopwords
warnings.filterwarnings('ignore')
random.seed(42)

DRIVE           = '/content/drive/MyDrive/emotion_augmentation/'
INPUT           = DRIVE + 'step0_extended.csv'
OUTPUT          = DRIVE + 'step2_eda.csv'
SBERT_THRESHOLD = 0.65
MIN_WORDS       = 6
EDA_N_AUG       = 4
ALPHA_SR        = 0.15
ALPHA_RI        = 0.10
ALPHA_RS        = 0.10
ALPHA_RD        = 0.10
STOP_WORDS      = set(stopwords.words('english'))
PROTECTED = {
    'happy','happiness','joy','excited','proud','elated','thrilled',
    'sad','sadness','grief','loss','disappointed','heartbroken','sorrow',
    'angry','anger','furious','rage','frustrated','irritated','outraged',
    'afraid','fear','scared','terrified','anxious','nervous','dread','panic',
    'disgusted','disgust','revolted','repulsed','contempt','nauseated',
    'surprised','shock','astonished','amazed','stunned','unexpected',
    'love','hate','hurt','pain','suffer','worry','stress','shame','guilty',
}
print('Config loaded')
print('Input :', INPUT)
print('Output:', OUTPUT)


Config loaded
Input : /content/drive/MyDrive/emotion_augmentation/step0_extended.csv
Output: /content/drive/MyDrive/emotion_augmentation/step2_eda.csv


In [ ]:
df = pd.read_csv(INPUT)
df['word_count'] = df['utterance'].str.split().str.len()
print('Loaded:', len(df), 'rows')
print(df['emotion_label'].value_counts().to_string())
sbert = SentenceTransformer('all-MiniLM-L6-v2')
print('SBERT ready')


Loaded: 7182 rows
emotion_label
Joy         1507
Disgust     1254
Sadness     1216
Fear        1129
Surprise    1045
Anger       1031


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT ready


In [ ]:
def get_synonyms(word):
    syns = []
    for syn in wordnet.synsets(word):
        for l in syn.lemmas():
            s = l.name().replace('_', ' ')
            if s.lower() != word.lower() and s.lower() not in PROTECTED:
                syns.append(s)
    return list(set(syns))

def synonym_replacement(sentence, n):
    words = sentence.split()
    new = words.copy()
    elig = [i for i,w in enumerate(words)
            if w.lower() not in STOP_WORDS and w.lower() not in PROTECTED and len(w)>3]
    random.shuffle(elig)
    r = 0
    for i in elig:
        syns = get_synonyms(words[i])
        if syns:
            new[i] = random.choice(syns)
            r += 1
            if r >= n: break
    return ' '.join(new)

def random_insertion(sentence, n):
    words = sentence.split()
    for _ in range(n):
        elig = [w for w in words if w.lower() not in STOP_WORDS and w.lower() not in PROTECTED]
        if not elig: break
        syns = get_synonyms(random.choice(elig))
        if syns:
            words.insert(random.randint(0, len(words)), random.choice(syns))
    return ' '.join(words)

def random_swap(sentence, n):
    words = sentence.split()
    if len(words) < 2: return sentence
    for _ in range(n):
        i, j = random.sample(range(len(words)), 2)
        if words[i].lower() not in PROTECTED and words[j].lower() not in PROTECTED:
            words[i], words[j] = words[j], words[i]
    return ' '.join(words)

def random_deletion(sentence, p):
    words = sentence.split()
    if len(words) <= 4: return sentence
    new = [w for w in words if w.lower() in PROTECTED or random.random() > p]
    return ' '.join(new) if len(new) >= 4 else sentence

EDA_OPS = [
    ('SR', lambda s: synonym_replacement(s, max(1, int(len(s.split())*ALPHA_SR)))),
    ('RI', lambda s: random_insertion(s,   max(1, int(len(s.split())*ALPHA_RI)))),
    ('RS', lambda s: random_swap(s,        max(1, int(len(s.split())*ALPHA_RS)))),
    ('RD', lambda s: random_deletion(s,    ALPHA_RD)),
]
print('EDA functions defined: SR, RI, RS, RD')


EDA functions defined: SR, RI, RS, RD


In [ ]:
eda_rows = []
log = {'total': 0, 'passed': 0, 'failed': 0}
print('Running EDA on', len(df), 'utterances...')

for idx, row in tqdm(df.iterrows(), total=len(df), desc='EDA'):
    orig_emb = sbert.encode([row['utterance']])
    count = 0
    for op_name, op_func in EDA_OPS:
        if count >= EDA_N_AUG: break
        try:
            aug = op_func(row['utterance'])
            log['total'] += 1
            if aug.lower() == row['utterance'].lower() or len(aug.split()) < MIN_WORDS:
                log['failed'] += 1
                continue
            aug_emb = sbert.encode([aug])
            sim = cosine_similarity(orig_emb, aug_emb)[0][0]
            if sim >= SBERT_THRESHOLD:
                new_row = row.to_dict()
                new_row['utterance']  = aug
                new_row['aug_method'] = 'eda_' + op_name
                new_row['word_count'] = len(aug.split())
                new_row['sbert_sim']  = round(float(sim), 3)
                eda_rows.append(new_row)
                log['passed'] += 1
                count += 1
            else:
                log['failed'] += 1
        except:
            log['failed'] += 1

df_eda = pd.DataFrame(eda_rows)
df_eda.to_csv(OUTPUT, index=False)
print('STEP 2 COMPLETE')
print('Generated :', log['passed'], 'utterances')
print('Pass rate :', round(log['passed']/max(1,log['total'])*100, 1), '%')
print(df_eda['emotion_label'].value_counts().to_string())
print(df_eda['aug_method'].value_counts().to_string())
print('Saved ->', OUTPUT)


Running EDA on 7182 utterances...


EDA: 100%|██████████| 7182/7182 [04:31<00:00, 26.45it/s]


STEP 2 COMPLETE
Generated : 26622 utterances
Pass rate : 92.7 %
emotion_label
Joy         5558
Sadness     4642
Disgust     4440
Fear        4193
Surprise    3895
Anger       3894
aug_method
eda_RS    7021
eda_RD    6589
eda_RI    6534
eda_SR    6478
Saved -> /content/drive/MyDrive/emotion_augmentation/step2_eda.csv
